# 🛠️ Notebook 2: Parking Lot — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/parking-lot
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from enum import Enum
from dataclasses import dataclass, field
import time, itertools

class Size(Enum):
    MOTORCYCLE = 1
    CAR = 2
    TRUCK = 3

@dataclass
class Vehicle:
    plate: str
    size: Size

class Spot:
    _ids = itertools.count(1)
    def __init__(self, size: Size):
        self.id = next(Spot._ids)
        self.size = size
        self.vehicle = None
    def can_fit(self, v: Vehicle) -> bool:
        return self.vehicle is None and v.size.value <= self.size.value
    def park(self, v: Vehicle):
        self.vehicle = v
    def leave(self):
        self.vehicle = None
    def __repr__(self):
        return f'Spot#{self.id}({self.size.name},{"free" if self.vehicle is None else self.vehicle.plate})'

class Level:
    def __init__(self, floor: int, spots: list[Spot]):
        self.floor = floor; self.spots = spots
    def find_spot(self, v: Vehicle):
        return next((s for s in self.spots if s.can_fit(v)), None)

@dataclass
class Ticket:
    vehicle: Vehicle
    spot: Spot
    entry_time: float = field(default_factory=time.time)

class ParkingLot:
    RATE_PER_HOUR = {Size.MOTORCYCLE:1, Size.CAR:2, Size.TRUCK:4}
    def __init__(self, levels: list[Level]):
        self.levels = levels
    def park(self, v: Vehicle):
        for lvl in self.levels:
            spot = lvl.find_spot(v)
            if spot:
                spot.park(v); return Ticket(v, spot)
        raise RuntimeError('lot full')
    def leave(self, t: Ticket, now=None) -> float:
        hours = max(1, ((now or time.time()) - t.entry_time)/3600)
        fee = self.RATE_PER_HOUR[t.vehicle.size] * hours
        t.spot.leave()
        return round(fee, 2)


## Walk-through

In [ ]:
lot = ParkingLot([
    Level(1, [Spot(Size.MOTORCYCLE), Spot(Size.CAR), Spot(Size.CAR), Spot(Size.TRUCK)]),
    Level(2, [Spot(Size.CAR), Spot(Size.CAR)]),
])

t1 = lot.park(Vehicle('BIKE-1', Size.MOTORCYCLE))
t2 = lot.park(Vehicle('CAR-1',  Size.CAR))
t3 = lot.park(Vehicle('TRUCK-1',Size.TRUCK))

for l in lot.levels:
    print(f'level {l.floor}:', l.spots)

# Simulate 2 hours parking for the car
fee = lot.leave(t2, now=t2.entry_time + 7200)
print(f'car paid ${fee}')


### Testing fit rules

In [ ]:
# A truck cannot fit in a car spot.
lot2 = ParkingLot([Level(1, [Spot(Size.CAR), Spot(Size.MOTORCYCLE)])])
try:
    lot2.park(Vehicle('BIG', Size.TRUCK))
except RuntimeError as e:
    print('expected error:', e)

# A motorcycle fits in a car spot when needed.
t = lot2.park(Vehicle('MOTO', Size.MOTORCYCLE))
print('parked:', t.spot)


### Extensions to try
- Reserved spots (handicapped, EV).
- Multiple entries / concurrency (threading locks).
- Monthly pass holders (different pricing).
- Dynamic pricing based on occupancy.